# HyperRAG V2 - Notebook 9: Chunk Size Comparison with Qwen3.5-9B

**Pipeline overview:**

Chunk sizes: `[512, 1024, 2048, 8192, 16384, 32768, full_page]`
K=5, expand_k=3, LLM=Qwen/Qwen3.5-9B

For each chunk size x 3 systems (Simple RAG, HtmlRAG, HyperRAG):
- Retrieval EM, Retrieval F1, Supporting Recall
- Qwen3.5-9B Gen EM, Gen F1

Results saved to `data/chunk_size_comparison/`

In [ ]:
# CONFIGURATION
N_FAIL = 'all'   # int or 'all'
N_BOTH = 'all'   # int or 'all'
CHUNK_SIZES   = [512, 1024, 2048, 8192, 16384, 32768, 'full_page']
CHUNK_OVERLAP = 64
K        = 5
EXPAND_K = 3
LOCAL_MODEL       = 'Qwen/Qwen3.5-9B'
MAX_CTX_CHARS_CAP = 16000
SAVE_RESULTS = True

subset_desc = f'{N_FAIL} graph-wins + {N_BOTH} both-pass'
if isinstance(N_FAIL, int) and isinstance(N_BOTH, int):
    subset_desc += f' = up to {N_FAIL + N_BOTH} questions'
else:
    subset_desc += ' = all available in each bucket'

print('=' * 62)
print('CONFIGURATION SUMMARY')
print('=' * 62)
print(f'  Subset     : {subset_desc}')
print(f'  Chunks     : {CHUNK_SIZES}')
print(f'  K={K}  expand_k={EXPAND_K}')
print(f'  LLM        : {LOCAL_MODEL}')
print(f'  Max ctx cap: {MAX_CTX_CHARS_CAP:,} chars')
print('=' * 62)

In [ ]:
import json, sys, re, string, csv
from pathlib import Path
from collections import Counter, defaultdict

import faiss
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if Path('utils.py').exists():
    BASE_DIR = Path('.')
elif Path('v2/utils.py').exists():
    BASE_DIR = Path('v2')
else:
    raise FileNotFoundError('utils.py not found. Run from v2/ or HyperRAG-M2/')

DATA_DIR   = BASE_DIR / 'data'
OUTPUT_DIR = DATA_DIR / 'chunk_size_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE_DIR))
from utils import (
    embed_texts, build_index, retrieve,
    compute_em, compute_f1, compute_supporting_recall, clean_html
)

print(f'BASE_DIR   : {BASE_DIR.resolve()}')
print(f'OUTPUT_DIR : {OUTPUT_DIR.resolve()}')
print('All imports OK.')

In [ ]:
print('Loading corpus and questions...')
with open(DATA_DIR / 'corpus_001.json')    as f: corpus    = json.load(f)
with open(DATA_DIR / 'questions_001.json') as f: questions = json.load(f)

for fname in ['results_simple_rag.json', 'results_htmlrag.json', 'results_hyperrag.json']:
    if not (DATA_DIR / fname).exists():
        raise FileNotFoundError(f'{fname} missing - run notebooks 02, 03, 04 first.')

with open(DATA_DIR / 'results_simple_rag.json') as f: simple_results = json.load(f)
with open(DATA_DIR / 'results_htmlrag.json')    as f: html_results   = json.load(f)
with open(DATA_DIR / 'results_hyperrag.json')   as f: hyper_results  = json.load(f)

simple_per_q = {r['id']: r for r in simple_results['per_question']}
html_per_q   = {r['id']: r for r in html_results['per_question']}
hyper_per_q  = {r['id']: r for r in hyper_results['per_question']}

print(f'Corpus    : {len(corpus):,} pages')
print(f'Questions : {len(questions):,}')
print()
print(f'  Simple RAG -- mean EM = {simple_results["mean_em"]:.3f}')
print(f'  HtmlRAG    -- mean EM = {html_results["mean_em"]:.3f}')
print(f'  HyperRAG   -- mean EM = {hyper_results["mean_em"]:.3f}')

In [ ]:
fail_then_graph = []
both_pass       = []
both_fail       = []
graph_fails     = []

for q in questions:
    sq = simple_per_q.get(q['id'])
    hq = hyper_per_q.get(q['id'])
    if not sq or not hq:
        continue
    s, h = int(sq['em']), int(hq['em'])
    if   s == 0 and h == 1: fail_then_graph.append(q)
    elif s == 1 and h == 1: both_pass.append(q)
    elif s == 0 and h == 0: both_fail.append(q)
    else:                   graph_fails.append(q)

n = len(questions)
print('EM Pattern Breakdown')
print(f'  graph-wins (S=0,H=1): {len(fail_then_graph):5d} ({100*len(fail_then_graph)/n:.1f}%)')
print(f'  both-pass  (S=1,H=1): {len(both_pass):5d} ({100*len(both_pass)/n:.1f}%)')
print()

if N_FAIL in ('all', None):
    n_fail = len(fail_then_graph)
else:
    n_fail = min(int(N_FAIL), len(fail_then_graph))

if N_BOTH in ('all', None):
    n_both = len(both_pass)
else:
    n_both = min(int(N_BOTH), len(both_pass))

subset_fail = fail_then_graph[:n_fail]
subset_both = both_pass[:n_both]
subset      = subset_fail + subset_both

print(f'Mixed subset: {n_fail} graph-wins + {n_both} both-pass = {len(subset)} questions')
print()
print(f'  {"Type":<12}  {"Gold Answer":<28}  Question')
print('-' * 85)
for q in subset[:4]:
    kind = 'graph-wins' if simple_per_q[q['id']]['em'] == 0.0 else 'both-pass'
    print(f'  {kind:<12}  {q["answer"][:26]:<28}  {q["question"][:48]}')

---
## Section 2 -- Chunking & Retrieval Systems

| Chunk size | Meaning |
|------------|----------|
| 512 -- 32768 | Fixed-size overlapping character-level chunks |
| `full_page` | Entire Wikipedia page = one chunk (no splitting) |

In [ ]:
def split_into_chunks(text: str, chunk_size: int, overlap: int = 64) -> list[str]:
    'Split text into overlapping chunks, breaking at word boundaries.'
    if len(text) <= chunk_size:
        return [text.strip()]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            last_space = text.rfind(' ', start, end)
            if last_space > start:
                end = last_space
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        nxt = end - overlap
        start = nxt if nxt > start else end
    return chunks


def build_chunk_corpus(corpus: list[dict], chunk_size: int, overlap: int = 64) -> list[dict]:
    'Convert page corpus into overlapping fixed-size chunk corpus.'
    docs = []
    for page in corpus:
        text_chunks = split_into_chunks(page['text'], chunk_size, overlap)
        raw_html    = page.get('html', '') or ''
        html_clean  = clean_html(raw_html) if raw_html else page['text']
        html_chunks = split_into_chunks(html_clean, chunk_size, overlap)
        n = len(text_chunks)
        for i, tc in enumerate(text_chunks):
            hc = html_chunks[i] if i < len(html_chunks) else tc
            docs.append({
                'title'    : page['title'],
                'chunk_idx': i,
                'n_chunks' : n,
                'text'     : tc,
                'html_text': hc,
                'links'    : page.get('links', []),
            })
    return docs


def build_full_page_corpus(corpus: list[dict]) -> list[dict]:
    'One chunk per page -- entire page text, no splitting.'
    docs = []
    for page in corpus:
        raw_html   = page.get('html', '') or ''
        html_clean = clean_html(raw_html) if raw_html else page['text']
        docs.append({
            'title'    : page['title'],
            'chunk_idx': 0,
            'n_chunks' : 1,
            'text'     : page['text'],
            'html_text': html_clean,
            'links'    : page.get('links', []),
        })
    return docs


def build_chunk_index(chunk_docs: list[dict]) -> faiss.IndexFlatIP:
    'Build FAISS cosine-similarity index over chunk plain texts.'
    vecs = embed_texts([d['text'] for d in chunk_docs])
    idx  = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return idx


def retrieve_chunks(query: str, chunk_index: faiss.IndexFlatIP,
                    chunk_docs: list[dict], k: int = 5) -> list[dict]:
    'Retrieve top-k chunks by cosine similarity.'
    q_vec    = embed_texts([query])
    k_actual = min(k, chunk_index.ntotal)
    scores, indices = chunk_index.search(q_vec, k_actual)
    results = []
    for ix, sc in zip(indices[0], scores[0]):
        if 0 <= ix < len(chunk_docs):
            c = dict(chunk_docs[ix])
            c['_score'] = float(sc)
            results.append(c)
    return results


def simple_rag_chunked(question: str, chunk_index, chunk_docs: list[dict], k: int = 5):
    'Simple RAG: retrieve top-k chunks, context = plain text.'
    chunks  = retrieve_chunks(question, chunk_index, chunk_docs, k=k)
    context = '\n\n'.join(c['text'] for c in chunks)
    titles  = list(dict.fromkeys(c['title'] for c in chunks))
    return context, titles, chunks


def html_rag_chunked(question: str, chunk_index, chunk_docs: list[dict], k: int = 5):
    'HtmlRAG: same retrieval, context uses cleaned HTML text.'
    chunks  = retrieve_chunks(question, chunk_index, chunk_docs, k=k)
    context = '\n\n'.join(c['html_text'] for c in chunks)
    titles  = list(dict.fromkeys(c['title'] for c in chunks))
    return context, titles, chunks


def hyper_rag_chunked(question: str, chunk_index, chunk_docs: list[dict],
                      corpus: list[dict], graph: nx.DiGraph,
                      k: int = 5, expand_k: int = 3):
    'HyperRAG: FAISS top-k + 1-hop graph neighbor expansion.'
    title_to_page   = {p['title']: p for p in corpus}
    initial         = retrieve_chunks(question, chunk_index, chunk_docs, k=k)
    init_ttls       = set(c['title'] for c in initial)
    neighbor_titles = set()
    for t in init_ttls:
        if graph.has_node(t):
            for nbr in list(graph.successors(t)) + list(graph.predecessors(t)):
                if nbr in title_to_page and nbr not in init_ttls:
                    neighbor_titles.add(nbr)
    expanded = []
    if neighbor_titles and expand_k > 0:
        nbr_chunks = [c for c in chunk_docs if c['title'] in neighbor_titles]
        if nbr_chunks:
            nv  = embed_texts([c['text'] for c in nbr_chunks])
            qv  = embed_texts([question])
            sc  = (qv @ nv.T)[0]
            top = sc.argsort()[::-1][:expand_k]
            for ix in top:
                ec = dict(nbr_chunks[ix])
                ec['_score']    = float(sc[ix])
                ec['_expanded'] = True
                expanded.append(ec)
    all_chunks = initial + expanded
    context    = '\n\n'.join(c['text'] for c in all_chunks)
    titles     = list(dict.fromkeys(c['title'] for c in all_chunks))
    return context, titles, all_chunks


print('Chunk + retrieval functions defined.')
print('  split_into_chunks, build_chunk_corpus, build_full_page_corpus')
print('  build_chunk_index, retrieve_chunks')
print('  simple_rag_chunked, html_rag_chunked, hyper_rag_chunked')

In [ ]:
def compute_retrieval_quality(context: str, chunks: list[dict],
                               gold_answer: str, gold_titles: list[str]) -> dict:
    'Compute retrieval EM, F1, supporting recall for retrieved chunks.'
    retrieved_titles = list(dict.fromkeys(c['title'] for c in chunks))
    ret_em  = float(compute_em(context, gold_answer))
    ret_f1  = round(float(compute_f1(context, gold_answer)), 4)
    sup_rec = round(float(compute_supporting_recall(retrieved_titles, gold_titles)), 4)
    chunk_details = []
    for rank, c in enumerate(chunks, 1):
        chunk_details.append({
            'rank'         : rank,
            'page_title'   : c['title'],
            'chunk_idx'    : c.get('chunk_idx', 0),
            'n_chunks'     : c.get('n_chunks', 1),
            'score'        : round(c.get('_score', 0.0), 4),
            'is_gold_page' : c['title'] in set(gold_titles),
            'has_answer'   : gold_answer.lower() in c['text'].lower(),
            'is_expanded'  : c.get('_expanded', False),
            'text_preview' : c['text'][:120].replace('\n', ' '),
        })
    return {
        'retrieval_em'      : ret_em,
        'retrieval_f1'      : ret_f1,
        'supporting_recall' : sup_rec,
        'retrieved_titles'  : retrieved_titles,
        'chunk_details'     : chunk_details,
    }

print('compute_retrieval_quality() defined.')

---
## Section 3 -- Metrics

| Metric | What it measures |
|--------|------------------|
| **Retrieval EM** | Gold answer found in retrieved context? (0 or 1) |
| **Retrieval F1** | Token overlap between retrieved context and gold answer |
| **Gen EM** | Exact match: Qwen's answer vs gold (normalized) |
| **Gen F1** | Token-overlap F1: Qwen's answer vs gold |

Retrieval metrics are LLM-independent -- they measure context quality only.

In [ ]:
def normalize_answer(text: str) -> str:
    'HotpotQA normalization: lowercase, strip articles and punctuation.'
    text = text.lower().strip()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())


def compute_gen_em(generated: str, gold: str) -> float:
    'Generation EM: 1 if normalized generated == normalized gold.'
    return 1.0 if normalize_answer(generated) == normalize_answer(gold) else 0.0


def compute_gen_f1(generated: str, gold: str) -> float:
    'Generation F1: token-overlap F1 between generated and gold.'
    gen_t  = normalize_answer(generated).split()
    gold_t = normalize_answer(gold).split()
    if not gen_t or not gold_t:
        return 0.0
    gen_c  = Counter(gen_t)
    gold_c = Counter(gold_t)
    overlap = sum((gen_c & gold_c).values())
    if overlap == 0:
        return 0.0
    prec = overlap / len(gen_t)
    rec  = overlap / len(gold_t)
    return round(2 * prec * rec / (prec + rec), 4)


print('normalize_answer(), compute_gen_em(), compute_gen_f1() defined.')

---
## Section 4 -- Qwen3.5-9B Local LLM Setup

We load `Qwen/Qwen3.5-9B` **once** and reuse it across all 7 chunk-size experiments.

- `device_map='auto'` handles GPU/CPU offloading for a 9B model
- ChatML format: `<|im_start|>system / user / assistant<|im_end|>`
- Thinking blocks `<think>...</think>` are stripped (Qwen3 hybrid mode support)

In [ ]:
import torch
from transformers import pipeline

device_str  = 'GPU (cuda)' if torch.cuda.is_available() else 'CPU'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'Loading: {LOCAL_MODEL}')
print(f'Device  : {device_str}  |  dtype={torch_dtype}')
print('First run downloads model weights -- may take a few minutes...')

local_llm = pipeline(
    'text-generation',
    model=LOCAL_MODEL,
    torch_dtype=torch_dtype,
    device_map='auto',
    trust_remote_code=True,
)
print('\nQwen LLM ready.')

_ctx    = 'The Eiffel Tower is in Paris, France. It was built in 1889.'
_q      = 'Where is the Eiffel Tower?'
_prompt = (
    '<|im_start|>system\nAnswer with a short phrase (1-5 words).<|im_end|>\n'
    f'<|im_start|>user\nContext: {_ctx}\nQuestion: {_q}<|im_end|>\n'
    '<|im_start|>assistant\n'
)
_raw = local_llm(_prompt, max_new_tokens=16, do_sample=False, return_full_text=False)[0]['generated_text']
_raw = re.sub(r'<think>.*?</think>', '', _raw, flags=re.DOTALL)
_ans = _raw.split('<|im_end|>')[0].split('<|endoftext|>')[0].strip()
print(f'Sanity check: "{_q}" -> "{_ans}"')

In [ ]:
def generate_qwen(question: str, context: str, max_ctx_chars: int = MAX_CTX_CHARS_CAP) -> str:
    'Generate a short answer with Qwen using ChatML prompt format.'
    ctx = context[:max_ctx_chars]
    prompt = (
        '<|im_start|>system\n'
        'You are a precise QA assistant. '
        'Answer the question with a short phrase (1-5 words). '
        'Extract the answer directly from the provided context.<|im_end|>\n'
        f'<|im_start|>user\nContext:\n{ctx}\n\nQuestion: {question}\n\n'
        'Answer (short phrase):<|im_end|>\n'
        '<|im_start|>assistant\n'
    )
    try:
        raw = local_llm(
            prompt, max_new_tokens=32, do_sample=False, return_full_text=False
        )[0]['generated_text']
        raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL)
        raw = raw.split('<|im_end|>')[0].split('<|endoftext|>')[0].strip()
        return raw
    except Exception as e:
        return f'[LLM error: {e}]'


print('generate_qwen() defined.')
print(f'  Max ctx cap: {MAX_CTX_CHARS_CAP:,} chars (scaled per chunk size in main loop)')

---
## Section 5 -- Main Comparison Loop

For each of the **7 chunk sizes**:
1. Build chunk corpus (or full-page corpus)
2. Embed all chunks -> fresh FAISS cosine index
3. Evaluate all questions x 3 systems:
   - Retrieve context, compute Retrieval EM / F1 / Supporting Recall
   - Generate with Qwen3.5-9B, compute Gen EM / Gen F1
4. Accumulate results in `all_results`

> **Runtime estimate:** Building 7 indexes + Qwen inference for N=40 questions
> takes ~2-6 hours on GPU. Reduce `N_FAIL`/`N_BOTH` for faster runs.

In [ ]:
try:
    from tqdm.auto import tqdm as tqdm_auto
except ImportError:
    def tqdm_auto(x, **kw): return x

# Build hyperlink graph once
print('Building hyperlink graph...')
all_titles = {p['title'] for p in corpus}
graph      = nx.DiGraph()
graph.add_nodes_from(all_titles)
for page in corpus:
    for link in page.get('links', []):
        if link in all_titles and link != page['title']:
            graph.add_edge(page['title'], link)
print(f'  Graph: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges')
print()

all_results = []
sys_names   = ['Simple RAG', 'HtmlRAG', 'HyperRAG']

for cs in CHUNK_SIZES:
    cs_label = 'full_page' if cs == 'full_page' else str(cs)
    print(f'\n{"=" * 65}')
    print(f'CHUNK SIZE: {cs_label}')
    print(f'{"=" * 65}')

    if cs == 'full_page':
        chunk_docs = build_full_page_corpus(corpus)
    else:
        chunk_docs = build_chunk_corpus(corpus, cs, CHUNK_OVERLAP)
    print(f'  {len(corpus):,} pages -> {len(chunk_docs):,} chunks '
          f'(avg {len(chunk_docs)/len(corpus):.1f} per page)')

    print('  Embedding + building FAISS index...')
    chunk_index = build_chunk_index(chunk_docs)
    print(f'  Index: {chunk_index.ntotal:,} vectors  dim=384')

    SYSTEMS = {
        'Simple RAG': lambda q, cd=chunk_docs, ci=chunk_index: (
            simple_rag_chunked(q['question'], ci, cd, k=K)
        ),
        'HtmlRAG': lambda q, cd=chunk_docs, ci=chunk_index: (
            html_rag_chunked(q['question'], ci, cd, k=K)
        ),
        'HyperRAG': lambda q, cd=chunk_docs, ci=chunk_index: (
            hyper_rag_chunked(q['question'], ci, cd, corpus, graph, k=K, expand_k=EXPAND_K)
        ),
    }

    max_ctx = MAX_CTX_CHARS_CAP if cs == 'full_page' else min(cs * K, MAX_CTX_CHARS_CAP)
    print(f'  Max ctx -> Qwen: {max_ctx:,} chars')
    print()

    for qi, q in enumerate(tqdm_auto(subset, desc=f'chunk={cs_label}')):
        gold   = q['answer']
        golds  = q['supporting_titles']
        q_type = 'graph-wins' if simple_per_q[q['id']]['em'] == 0.0 else 'both-pass'

        for sys_name, fn in SYSTEMS.items():
            ctx, titles, chunks = fn(q)
            quality = compute_retrieval_quality(ctx, chunks, gold, golds)
            qwen_ans    = generate_qwen(q['question'], ctx, max_ctx_chars=max_ctx)
            qwen_gen_em = compute_gen_em(qwen_ans, gold)
            qwen_gen_f1 = compute_gen_f1(qwen_ans, gold)

            all_results.append({
                'chunk_size'        : cs_label,
                'question_id'       : q['id'],
                'question'          : q['question'],
                'gold_answer'       : gold,
                'supporting_titles' : list(golds),
                'subset_type'       : q_type,
                'system'            : sys_name,
                'k'                 : K,
                'expand_k'          : EXPAND_K if sys_name == 'HyperRAG' else 0,
                'retrieval_em'      : quality['retrieval_em'],
                'retrieval_f1'      : quality['retrieval_f1'],
                'supporting_recall' : quality['supporting_recall'],
                'retrieved_titles'  : quality['retrieved_titles'],
                'qwen_answer'       : qwen_ans,
                'qwen_gen_em'       : qwen_gen_em,
                'qwen_gen_f1'       : qwen_gen_f1,
            })

        if (qi + 1) % 10 == 0 or qi == len(subset) - 1:
            last_hyp = [r for r in all_results if r['system'] == 'HyperRAG'][-1]
            print(f'  [{qi+1}/{len(subset)}] Q: "{q["question"][:52]}"')
            print(f'    Gold: "{gold}"  | Type: {q_type}')
            print(f'    [HyperRAG] Ret.EM={last_hyp["retrieval_em"]:.0f}  '
                  f'Ret.F1={last_hyp["retrieval_f1"]:.4f}  '
                  f'GenEM={last_hyp["qwen_gen_em"]:.0f}  '
                  f'GenF1={last_hyp["qwen_gen_f1"]:.3f}')
            print()

    print(f'chunk={cs_label} done - {len(subset) * len(SYSTEMS)} rows added.')

print(f'\n{"=" * 65}')
print('ALL CHUNK SIZES COMPLETE')
print(f'Total: {len(all_results)} result rows')
print(f'  ({len(CHUNK_SIZES)} chunk sizes x {len(subset)} questions x {len(sys_names)} systems)')

---
## Section 6 -- Save Results

In [ ]:
cs_labels = [str(cs) for cs in CHUNK_SIZES]

def mean_safe(rows, key):
    vals = [r[key] for r in rows if r.get(key) is not None]
    return float(np.mean(vals)) if vals else 0.0


if SAVE_RESULTS:
    # JSON
    json_path = OUTPUT_DIR / 'chunk_comparison_results.json'
    out = {
        'config': {
            'chunk_sizes'      : cs_labels,
            'chunk_overlap'    : CHUNK_OVERLAP,
            'k'                : K,
            'expand_k'         : EXPAND_K,
            'local_model'      : LOCAL_MODEL,
            'max_ctx_chars_cap': MAX_CTX_CHARS_CAP,
            'n_questions'      : len(subset),
        },
        'results': all_results,
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2, ensure_ascii=False)
    print(f'JSON saved  ->  {json_path}  ({json_path.stat().st_size/1024:.0f} KB)')

    # CSV 1: per-question flat
    csv_path    = OUTPUT_DIR / 'chunk_comparison_per_question.csv'
    flat_fields = [
        'chunk_size', 'question_id', 'subset_type', 'system',
        'question', 'gold_answer',
        'retrieval_em', 'retrieval_f1', 'supporting_recall',
        'retrieved_titles',
        'qwen_answer', 'qwen_gen_em', 'qwen_gen_f1',
    ]
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=flat_fields, extrasaction='ignore')
        writer.writeheader()
        for r in all_results:
            row = dict(r)
            row['retrieved_titles'] = ' | '.join(r.get('retrieved_titles', []))
            writer.writerow(row)
    print(f'CSV 1 saved ->  {csv_path}  ({len(all_results)} rows)')

    # CSV 2: aggregate summary
    agg_path = OUTPUT_DIR / 'chunk_comparison_summary.csv'
    agg_rows = []
    for csl in cs_labels:
        for sys_name in sys_names:
            rows = [r for r in all_results if r['chunk_size'] == csl and r['system'] == sys_name]
            if not rows:
                continue
            agg_rows.append({
                'chunk_size'             : csl,
                'system'                 : sys_name,
                'n_questions'            : len(rows),
                'mean_retrieval_em'      : round(mean_safe(rows, 'retrieval_em'),      4),
                'mean_retrieval_f1'      : round(mean_safe(rows, 'retrieval_f1'),      4),
                'mean_supporting_recall' : round(mean_safe(rows, 'supporting_recall'), 4),
                'mean_qwen_gen_em'       : round(mean_safe(rows, 'qwen_gen_em'),       4),
                'mean_qwen_gen_f1'       : round(mean_safe(rows, 'qwen_gen_f1'),       4),
            })
    with open(agg_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(agg_rows[0].keys()))
        writer.writeheader()
        writer.writerows(agg_rows)
    print(f'CSV 2 saved ->  {agg_path}  ({len(agg_rows)} rows)')
    print()
    print(f'All outputs in: {OUTPUT_DIR.resolve()}')
else:
    print('SAVE_RESULTS=False - skipping.')
    cs_labels = [str(cs) for cs in CHUNK_SIZES]
    def mean_safe(rows, key):
        vals = [r[key] for r in rows if r.get(key) is not None]
        return float(np.mean(vals)) if vals else 0.0

---
## Section 7 -- Visualizations

Three charts:
1. **2x2 line plots** -- Ret.EM, Ret.F1, Gen.EM, Gen.F1 vs chunk size
2. **Grouped bar charts** -- Gen EM & Gen F1 per chunk size
3. **Extraction gap** -- Retrieval EM vs Gen EM per system

In [ ]:
def get_means(metric: str, sys_name: str) -> list[float]:
    'Mean of metric for each chunk size label.'
    return [
        mean_safe(
            [r for r in all_results if r['chunk_size'] == csl and r['system'] == sys_name],
            metric
        )
        for csl in cs_labels
    ]


sys_colors  = {'Simple RAG': 'steelblue', 'HtmlRAG': 'teal', 'HyperRAG': 'coral'}
sys_markers = {'Simple RAG': 'o', 'HtmlRAG': 's', 'HyperRAG': '^'}

metrics_info = [
    ('retrieval_em',  'Retrieval EM\n(gold in context?)'),
    ('retrieval_f1',  'Retrieval F1\n(token overlap)'),
    ('qwen_gen_em',   'Qwen Gen EM\n(exact match)'),
    ('qwen_gen_f1',   'Qwen Gen F1\n(token F1)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
fig.suptitle(
    f'HyperRAG -- Chunk Size Impact on Retrieval & Generation Quality\n'
    f'(K={K}, expand_k={EXPAND_K}  |  {len(subset)} questions  |  LLM: {LOCAL_MODEL.split("/")[-1]})',
    fontsize=13, fontweight='bold',
)

for ax, (metric, title) in zip(axes, metrics_info):
    for sys_name in sys_names:
        vals = get_means(metric, sys_name)
        ax.plot(cs_labels, vals, marker=sys_markers[sys_name],
                color=sys_colors[sys_name], label=sys_name,
                linewidth=2.2, markersize=8)
        for xi, v in enumerate(vals):
            ax.annotate(f'{v:.3f}', (xi, v), textcoords='offset points',
                        xytext=(0, 9), ha='center', fontsize=7,
                        color=sys_colors[sys_name], fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Chunk Size (chars)', fontsize=9)
    ax.set_xticks(range(len(cs_labels)))
    ax.set_xticklabels(cs_labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 1.18)
    ax.set_ylabel(title.split('\n')[0], fontsize=9)
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(axis='y', alpha=0.3)

handles = [mpatches.Patch(color=sys_colors[s], label=s) for s in sys_names]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
chart1_path = OUTPUT_DIR / 'chunk_size_line_plots.png'
plt.savefig(str(chart1_path), dpi=130, bbox_inches='tight')
plt.show()
print(f'Chart 1 saved -> {chart1_path}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'Qwen3.5-9B Generation Quality by Chunk Size & System\n'
    f'(K={K}, expand_k={EXPAND_K}  |  {len(subset)} questions)',
    fontsize=12, fontweight='bold',
)
x       = np.arange(len(cs_labels))
width   = 0.25
offsets = [-width, 0, width]

for ax_idx, (metric, ylabel) in enumerate([
    ('qwen_gen_em', 'Generation EM (Exact Match)'),
    ('qwen_gen_f1', 'Generation F1 (Token F1)'),
]):
    ax = axes[ax_idx]
    for i, sys_name in enumerate(sys_names):
        vals = get_means(metric, sys_name)
        bars = ax.bar(x + offsets[i], vals, width,
                      label=sys_name, color=sys_colors[sys_name], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0.005:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                        f'{h:.3f}', ha='center', va='bottom',
                        fontsize=6.5, fontweight='bold')
    ax.set_title(ylabel, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(cs_labels, rotation=30, ha='right', fontsize=8)
    ax.set_xlabel('Chunk Size (chars)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_ylim(0, 1.18)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
chart2_path = OUTPUT_DIR / 'chunk_size_grouped_bars.png'
plt.savefig(str(chart2_path), dpi=130, bbox_inches='tight')
plt.show()
print(f'Chart 2 saved -> {chart2_path}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(
    'Retrieval EM vs Qwen Generation EM -- Extraction Gap by System\n'
    f'(K={K}, expand_k={EXPAND_K}  |  {len(subset)} questions)',
    fontsize=12, fontweight='bold',
)
x     = np.arange(len(cs_labels))
width = 0.35

for ax, sys_name in zip(axes, sys_names):
    ret_vals = get_means('retrieval_em', sys_name)
    gen_vals = get_means('qwen_gen_em',  sys_name)
    ax.bar(x - width/2, ret_vals, width, label='Retrieval EM', color='#aec6cf', alpha=0.9)
    ax.bar(x + width/2, gen_vals, width, label='Qwen Gen EM',  color=sys_colors[sys_name], alpha=0.85)
    for xi, (rv, gv) in enumerate(zip(ret_vals, gen_vals)):
        ax.text(xi - width/2, rv + 0.012, f'{rv:.2f}', ha='center', va='bottom', fontsize=6.5)
        ax.text(xi + width/2, gv + 0.012, f'{gv:.2f}', ha='center', va='bottom', fontsize=6.5)
    ax.set_title(sys_name, fontsize=11, fontweight='bold', color=sys_colors[sys_name])
    ax.set_xticks(x)
    ax.set_xticklabels(cs_labels, rotation=30, ha='right', fontsize=7)
    ax.set_xlabel('Chunk Size', fontsize=9)
    ax.set_ylabel('EM' if sys_name == sys_names[0] else '')
    ax.set_ylim(0, 1.18)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
chart3_path = OUTPUT_DIR / 'retrieval_vs_generation_gap.png'
plt.savefig(str(chart3_path), dpi=130, bbox_inches='tight')
plt.show()
print(f'Chart 3 saved -> {chart3_path}')

In [ ]:
def fmt(v): return f'{v:.3f}'

print('AGGREGATE RESULTS -- Chunk Size x System')
print('=' * 100)
hdr = f'  {"Chunk":<12} {"System":<13} {"Ret.EM":>7} {"Ret.F1":>7} {"SuppRec":>8} {"GenEM":>7} {"GenF1":>7}'
print(hdr)
print(f'  {"-" * 63}')

for csl in cs_labels:
    for sys_name in sys_names:
        rows = [r for r in all_results if r['chunk_size'] == csl and r['system'] == sys_name]
        if not rows:
            continue
        print(f'  {csl:<12} {sys_name:<13} '
              f'{fmt(mean_safe(rows, "retrieval_em")):>7} '
              f'{fmt(mean_safe(rows, "retrieval_f1")):>7} '
              f'{fmt(mean_safe(rows, "supporting_recall")):>8} '
              f'{fmt(mean_safe(rows, "qwen_gen_em")):>7} '
              f'{fmt(mean_safe(rows, "qwen_gen_f1")):>7}')
    print()

---
## Summary

### Output Files (`data/chunk_size_comparison/`)

| File | Contents |
|------|----------|
| `chunk_comparison_results.json` | Full results: config + all per-question rows |
| `chunk_comparison_per_question.csv` | Flat table: one row per (chunk_size x question x system) |
| `chunk_comparison_summary.csv` | Aggregate means per (chunk_size x system) |
| `chunk_size_line_plots.png` | 2x2 line plots: all 4 metrics vs chunk size |
| `chunk_size_grouped_bars.png` | Grouped bars: Gen EM & F1 per chunk size |
| `retrieval_vs_generation_gap.png` | Extraction gap: Retrieval EM vs Gen EM |

### Interpretation

| Observation | Meaning |
|-------------|----------|
| Retrieval EM rises with chunk size | Larger chunks more likely to contain gold answer |
| Gen EM < Retrieval EM | LLM extraction gap -- answer in context but LLM misses it |
| HyperRAG advantage at small chunks | Graph expansion compensates for narrow retrieval window |
| Gen EM may drop at very large chunks | Very long contexts can dilute LLM attention |